In [ ]:
import json

import pandas as pd
import ast

from typing import Union, List

/home/yishin/miniconda3/envs/patent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Impact DF

In [ ]:
def try_literal_eval(x):
    if isinstance(x, str):
        x = x.strip()
        if (x.startswith("[") and x.endswith("]")) or \
           (x.startswith("{") and x.endswith("}")) or \
           (x.startswith("(") and x.endswith(")")):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x

In [ ]:
Impact_df = pd.read_csv("Impact_2022.csv",encoding="utf-8")
Impact_df = Impact_df.map(try_literal_eval)

Impact_df.head()

,title,file_names,Loc_class,main_class,sub_class
0,Data reader,[impact_dataset/2022/USD0949851-20220426/USD09...,{14-02},14,2
1,Panel light,[impact_dataset/2022/USD0971479-20221129/USD09...,{26-05},26,5
2,Massager,[impact_dataset/2022/USD0959008-20220726/USD09...,{24-01},24,1
3,Luggage,[impact_dataset/2022/USD0965975-20221011/USD09...,"{07-05, 03-01}",7,5
4,Wall-mounted safe,[impact_dataset/2022/USD0942735-20220201/USD09...,"{25-02, 06-04}",25,2


# Preprocessing

## USPC/Locarno

### USPC Normalization

In [5]:
with open(conversion_dir / "USPC_RANGES.json", "r", encoding="utf-8") as f:
    USPC_RANGES = json.load(f)

In [6]:
def normalize_code(code: Union[str, List[str]]) -> set[str]:
    results = set()

    # ---- normalize input into a list of strings ----
    if isinstance(code, str):
        codes_to_process = code.replace(" ", "").split(",")
    else:
        # list of strings → clean each, split commas if present
        codes_to_process = []
        for c in code:
            if isinstance(c, str):
                codes_to_process.extend(c.replace(" ", "").split(","))

    # ---- core normalization logic ----
    for single_code in codes_to_process:
        for prefix_len in (2, 3):
            if len(single_code) <= prefix_len:
                continue

            prefix = single_code[:prefix_len]
            suffix = single_code[prefix_len:]

            # reject leading-zero suffixes
            if len(suffix) > 1 and suffix.startswith("0"):
                continue

            ranges = USPC_RANGES.get(prefix)
            if not ranges:
                continue

            try:
                suffix_int = int(suffix)
            except ValueError:
                continue

            start, end = ranges
            if start <= suffix_int < end:
                results.add(f"{prefix}-{suffix}")

    return results

In [7]:
Impact_df["USPC_class"] = Impact_df["class"].apply(normalize_code)
Impact_df["USPC_class_search"] = Impact_df["class_search"].apply(normalize_code)

Impact_df.head()

,year,date,title,caption,file_names,fig_desc,class,class_search,USPC_class,USPC_class_search
0,2021,20211207,Display screen or portion thereof with a graph...,"The image is a square, and it displays a graph...",[impact_dataset/2021/USD0937859-20211207/USD09...,[The FIGURE is a from view of a display screen...,D14486,"[1404, D14486, D14486, D14486, D14485, D14492,...",{D14-486},"{D14-486, D14-492, D14-489, D14-485, D14-494, ..."
1,2021,20210126,Garment with a side pocket,The image is a square-shaped illustration of a...,[impact_dataset/2021/USD0908314-20210126/USD09...,[FIG. 1 is a front left perspective view of th...,"D 2728, D2840","[0202, D 2728, D 2839, D 2829, D 2750, D 2839,...","{D2-728, D2-840, D28-40}","{D2-728, D21-804, D2-865, D2-873, D2-720, D28-..."
2,2021,20210406,Quilted fabric,"The image is a square shape, and its functiona...",[impact_dataset/2021/USD0915081-20210406/USD09...,[A portion of the disclosure of this patent do...,D 5 59,"[0505, D 5 59, D 5 99, D 5 53, D 5 62, D 5 63,...",{D5-59},"{D6-613, D5-1, D20-27, D32-57, D3-240, D3-257,..."
3,2021,20210316,Display screen or portion thereof with animate...,The image is a square-shaped display screen wi...,[impact_dataset/2021/USD0913312-20210316/USD09...,[FIG. 1 is a front view of a display screen or...,"D14486,D14487","[1404, D14486, D14486, D14485, D14486, D14488,...","{D14-486, D14-487}","{D14-486, D14-485, D14-488, D14-487}"
4,2021,20210907,Bat,"The image is a long, thin, and elongated shape...",[impact_dataset/2021/USD0930094-20210907/USD09...,[FIG. 1 is a perspective view of a bat showing...,D21725,"[2102, D21725, 473568, D21725, D21725, 473568,...",{D21-725},"{D21-722, D21-756, D8-303, D21-753, D21-725}"


### USPC to Locarno Conversion

In [8]:
with open(conversion_dir / "USPC_CONVERSION_CHART.json", "r", encoding="utf-8") as f:
    CONVERSION_CHART = json.load(f)

In [9]:
def convert_USPC_Locarno(USPC_Codes: set):
    Locarno_Codes = []

    for i in USPC_Codes:
        try:
            main_class, subclass = i.split("-")
        except ValueError:
            return set()
        
        for cat in CONVERSION_CHART[main_class]:
            bound = cat["U.S. Subclass"].replace(" ", "").split("-")

            if len(bound) == 1:
                if subclass == bound[0]:
                    Locarno_Codes.append(cat["Locarno Class - Subclass"].replace(" ", ""))
            else:
                if float(bound[0]) <= float(subclass) <= float(bound[1]):
                    Locarno_Codes.append(cat["Locarno Class - Subclass"].replace(" ", ""))

    return set(Locarno_Codes)

In [10]:
Impact_df["Loc_class"] = Impact_df["USPC_class"].apply(convert_USPC_Locarno)
Impact_df["Loc_class_search"] = Impact_df["USPC_class_search"].apply(convert_USPC_Locarno)

Impact_df.head()

,year,date,title,caption,file_names,fig_desc,class,class_search,USPC_class,USPC_class_search,Loc_class,Loc_class_search
0,2021,20211207,Display screen or portion thereof with a graph...,"The image is a square, and it displays a graph...",[impact_dataset/2021/USD0937859-20211207/USD09...,[The FIGURE is a from view of a display screen...,D14486,"[1404, D14486, D14486, D14486, D14485, D14492,...",{D14-486},"{D14-486, D14-492, D14-489, D14-485, D14-494, ...",{14-04},{14-04}
1,2021,20210126,Garment with a side pocket,The image is a square-shaped illustration of a...,[impact_dataset/2021/USD0908314-20210126/USD09...,[FIG. 1 is a front left perspective view of th...,"D 2728, D2840","[0202, D 2728, D 2839, D 2829, D 2750, D 2839,...","{D2-728, D2-840, D28-40}","{D2-728, D21-804, D2-865, D2-873, D2-720, D28-...","{28-06, 02-02}","{29-02, 28-03, 02-03, 02-01, 28-06, 02-02}"
2,2021,20210406,Quilted fabric,"The image is a square shape, and its functiona...",[impact_dataset/2021/USD0915081-20210406/USD09...,[A portion of the disclosure of this patent do...,D 5 59,"[0505, D 5 59, D 5 99, D 5 53, D 5 62, D 5 63,...",{D5-59},"{D6-613, D5-1, D20-27, D32-57, D3-240, D3-257,...",{05-03},"{06-13, 29-01, 06-04, 20-03, 32-01, 07-05, 05-..."
3,2021,20210316,Display screen or portion thereof with animate...,The image is a square-shaped display screen wi...,[impact_dataset/2021/USD0913312-20210316/USD09...,[FIG. 1 is a front view of a display screen or...,"D14486,D14487","[1404, D14486, D14486, D14485, D14486, D14488,...","{D14-486, D14-487}","{D14-486, D14-485, D14-488, D14-487}",{14-04},{14-04}
4,2021,20210907,Bat,"The image is a long, thin, and elongated shape...",[impact_dataset/2021/USD0930094-20210907/USD09...,[FIG. 1 is a perspective view of a bat showing...,D21725,"[2102, D21725, 473568, D21725, D21725, 473568,...",{D21-725},"{D21-722, D21-756, D8-303, D21-753, D21-725}",{21-02},"{08-06, 21-02}"


### Empty Check

In [11]:
def is_class_empty(x):
    return (
        pd.isna(x)
        or isinstance(x, str)
        or (isinstance(x, set) and len(x) == 0)
    )

In [12]:
mask = (
    Impact_df["USPC_class"].apply(is_class_empty)
    &
    Impact_df["Loc_class"].apply(is_class_empty)
)
Impact_df = Impact_df[~mask]

In [13]:
def to_set_or_empty(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    if isinstance(x, (list, tuple, set)):
        return set(x)
    return {str(x)}

In [14]:
row_has_empty_mask = Impact_df.apply(
    lambda row: any(len(to_set_or_empty(v)) == 0 for v in row),
    axis=1,
)

In [15]:
row_has_empty_mask.sum()
Impact_df = Impact_df[~row_has_empty_mask].reset_index(drop=True)
Impact_df.shape[0]

91066

## Filtering

In [ ]:
Impact_2022_df = Impact_df[Impact_df["year"] == 2022]

Impact_2022_df = Impact_2022_df[["title", "file_names", "Loc_class"]]

Impact_2022_df = Impact_2022_df.reset_index(drop=True)

Impact_2022_df.head(10)

,title,file_names,Loc_class
0,Data reader,[impact_dataset/2022/USD0949851-20220426/USD09...,{14-02}
1,Panel light,[impact_dataset/2022/USD0971479-20221129/USD09...,{26-05}
2,Massager,[impact_dataset/2022/USD0959008-20220726/USD09...,{24-01}
3,Luggage,[impact_dataset/2022/USD0965975-20221011/USD09...,"{07-05, 03-01}"
4,Wall-mounted safe,[impact_dataset/2022/USD0942735-20220201/USD09...,"{25-02, 06-04}"
5,Pair of earrings,[impact_dataset/2022/USD0964203-20220920/USD09...,{11-01}
6,Razor hanger,[impact_dataset/2022/USD0964065-20220920/USD09...,{06-04}
7,Toy,[impact_dataset/2022/USD0942553-20220201/USD09...,{21-01}
8,Single shot protection device,[impact_dataset/2022/USD0947978-20220405/USD09...,{22-01}
9,Pet feeding station,[impact_dataset/2022/USD0970824-20221122/USD09...,{30-03}


## Class Dissection

In [44]:
def split_class(value):
    # Handle empty sets
    if not value:
        return pd.Series([None, None])

    # Convert set to list and take first item
    first_item = list(value)[0]

    # Split x-y format
    main_class, sub_class = first_item.split("-")

    return pd.Series([main_class, sub_class])

In [45]:
Impact_df[["main_class", "sub_class"]] = (
    Impact_df["Loc_class"].apply(split_class)
)

In [52]:
Impact_df.head(10)

,title,file_names,Loc_class,main_class,sub_class
0,Data reader,[impact_dataset/2022/USD0949851-20220426/USD09...,{14-02},14,2
1,Panel light,[impact_dataset/2022/USD0971479-20221129/USD09...,{26-05},26,5
2,Massager,[impact_dataset/2022/USD0959008-20220726/USD09...,{24-01},24,1
3,Luggage,[impact_dataset/2022/USD0965975-20221011/USD09...,"{07-05, 03-01}",7,5
4,Wall-mounted safe,[impact_dataset/2022/USD0942735-20220201/USD09...,"{25-02, 06-04}",25,2
5,Pair of earrings,[impact_dataset/2022/USD0964203-20220920/USD09...,{11-01},11,1
6,Razor hanger,[impact_dataset/2022/USD0964065-20220920/USD09...,{06-04},6,4
7,Toy,[impact_dataset/2022/USD0942553-20220201/USD09...,{21-01},21,1
8,Single shot protection device,[impact_dataset/2022/USD0947978-20220405/USD09...,{22-01},22,1
9,Pet feeding station,[impact_dataset/2022/USD0970824-20221122/USD09...,{30-03},30,3


## Save

In [51]:
Impact_df.to_csv("Impact_2022_LLaVa_NoKW.csv", index=False, encoding="utf-8")

# Subsampling

In [72]:
Impact_df = pd.read_csv("Impact_2022.csv",encoding="utf-8")
Impact_df = Impact_df.map(try_literal_eval)

Impact_df.head()

,title,file_names,Loc_class,main_class,sub_class
0,Data reader,[impact_dataset/2022/USD0949851-20220426/USD09...,{14-02},14,2
1,Panel light,[impact_dataset/2022/USD0971479-20221129/USD09...,{26-05},26,5
2,Massager,[impact_dataset/2022/USD0959008-20220726/USD09...,{24-01},24,1
3,Luggage,[impact_dataset/2022/USD0965975-20221011/USD09...,"{07-05, 03-01}",7,5
4,Wall-mounted safe,[impact_dataset/2022/USD0942735-20220201/USD09...,"{25-02, 06-04}",25,2


In [73]:
len(Impact_df)

30868

In [74]:
class_column = "main_class"

In [75]:
sample_df = (
    Impact_df
    .groupby(class_column, group_keys=False)
    .sample(frac=0.1, random_state=42)
    .reset_index(drop=True)
)

sample_df.head(10)

,title,file_names,Loc_class,main_class,sub_class
0,Article of footwear,[impact_dataset/2022/USD0943877-20220222/USD09...,{02-04},2,4
1,Shoe,[impact_dataset/2022/USD0950931-20220510/USD09...,{02-04},2,4
2,Rear combination lamp for automobile,[impact_dataset/2022/USD0957706-20220712/USD09...,"{02-07, 26-06}",2,7
3,Portable light beacon,[impact_dataset/2022/USD0959716-20220802/USD09...,"{02-07, 26-02}",2,7
4,Water shoe,[impact_dataset/2022/USD0960535-20220816/USD09...,{02-04},2,4
5,Shoe,[impact_dataset/2022/USD0955707-20220628/USD09...,{02-04},2,4
6,Vehicle center high mount brake light,[impact_dataset/2022/USD0940933-20220111/USD09...,"{02-07, 26-06}",2,7
7,Vehicle headlight,[impact_dataset/2022/USD0959711-20220802/USD09...,"{02-07, 26-06}",2,7
8,Running light,[impact_dataset/2022/USD0940367-20220104/USD09...,"{02-07, 26-02}",2,7
9,Shoe,[impact_dataset/2022/USD0970192-20221122/USD09...,{02-04},2,4


In [87]:
len(sample_df)

3084

In [77]:
print(sample_df.columns.tolist())

['title', 'file_names', 'Loc_class', 'main_class', 'sub_class']


In [78]:
print(sample_df[class_column].value_counts())

main_class
14    482
12    244
24    241
2     202
23    198
21    167
13    149
6     142
9     140
8     139
15    127
7     117
26     99
11     78
10     76
28     76
16     72
25     61
30     42
27     37
4      34
3      33
22     32
19     26
29     24
18     16
20     11
31      9
17      7
5       3
Name: count, dtype: int64


In [95]:
sample_df.to_csv("Impact_2022_Subsample.csv", index=False, encoding="utf-8")